# Second Part: Data cleaning and dashboard preparation

This notebook converts the filtered CORDIS and CTIS datasets into stable, dashboard-ready tables. It standardizes data types, removes export artifacts, adds readable country names, and normalizes the many-country clinical-trial location field.

## Outputs

- `data_final/ConsurDatabase_clean.csv`: one row per project–organisation relationship.
- `data_final/TrialsDatabase_clean.csv`: one row per clinical trial.
- `data_final/TrialCountries.csv`: one row per unique trial–country relationship.


## 1. Imports and paths

In [1]:
import sys
sys.executable
import sys
!{sys.executable} -m pip install babel

In [2]:
from pathlib import Path
import re

import pandas as pd
from babel import Locale

In [3]:
cordis_path = Path("../../_local_data_backup/CordisDatabase.csv")
trials_path = Path("data/data_final/TrialsDatabase.csv")

df_cordis = pd.read_csv(cordis_path)
df_trials = pd.read_csv(trials_path)

/var/folders/_z/vq1q1pzj639b28jbhlxzd0zw0000gn/T/ipykernel_22342/1358514559.py:4: DtypeWarning: Columns (0: active) have mixed types. Specify dtype option on import or set low_memory=False.
  df_cordis = pd.read_csv(cordis_path)


## 2. Reusable cleaning functions

In [4]:
ENGLISH = Locale("en")
COUNTRY_OVERRIDES = {
    "EL": "Greece",      
    "UK": "United Kingdom",  
    "XK": "Kosovo",
    "ZZ": "Unknown",
}

EUROPE_CODES = {
    "AL", "AD", "AT", "BY", "BE", "BA", "BG", "HR", "CY", "CZ", "DK", "EE",
    "FI", "FR", "DE", "EL", "HU", "IS", "IE", "IT", "XK", "LV", "LI", "LT",
    "LU", "MT", "MD", "MC", "ME", "NL", "MK", "NO", "PL", "PT", "RO", "RU",
    "SM", "RS", "SK", "SI", "ES", "SE", "CH", "TR", "UA", "UK", "VA"
}

AMERICAS_CODES = {
    "US", "CA", "MX", "BR", "AR", "CL", "CO", "PE", "UY", "PY", "BO", "EC",
    "VE", "CR", "PA", "GT", "HN", "SV", "NI", "DO", "CU", "PR"
}

ASIA_CODES = {
    "CN", "JP", "KR", "IN", "SG", "MY", "TH", "VN", "PH", "ID", "IL", "AE",
    "SA", "QA", "KW", "TW", "HK"
}

AFRICA_CODES = {
    "ZA", "EG", "MA", "TN", "NG", "KE", "GH", "ET", "UG", "SN", "CG", "GW"
}

OCEANIA_CODES = {"AU", "NZ"}



category_keywords = {
    "Automation": [
        "laboratory automation", "lab automation", "liquid handling",
        "sample preparation", "workflow automation"
    ],

    "Genomics / Molecular Biology": [
        "genomics", "genomic", "sequencing", "next generation sequencing",
        "ngs", "pcr", "qpcr", "digital pcr", "rna", "dna",
        "transcriptomics", "multi-omics", "multiomics"
    ],

    "Proteomics / Multi-omics": [
        "proteomics", "proteomic", "metabolomics"
    ],

    "Diagnostics / Biomarkers": [
        "biomarker", "biomarkers", "diagnostics", "molecular diagnostics",
        "clinical diagnostics", "ivd", "in vitro diagnostics",
        "liquid biopsy", "companion diagnostics", "genetic testing",
        "molecular profiling", "pathology", "molecular pathology"
    ],

    "Oncology": [
        "oncology", "cancer", "tumor", "tumour", "leukemia",
        "leukaemia", "lymphoma", "carcinoma", "solid tumor", "solid tumour"
    ],

    "Cell / Advanced Therapies": [
        "cell therapy", "cart", "car-t", "t cell", "t-cell",
        "stem cell", "cell biology", "cellomics", "organoids",
        "3d cell", "cell and tissue", "single cell"
    ],

    "Pharma / Biotech": [
        "drug discovery", "drug development", "biopharma",
        "biotechnology", "personalized medicine", "precision medicine"
    ],

    "Digital Health / Bioinformatics": [
        "bioinformatics", "data-driven", "digital health"
    ],

    "MedTech / Medical Device": [
        "medtech", "medical device", "microfluidics"
    ],

}

In [5]:
def get_macro_region(country_code):
    if country_code in EUROPE_CODES:
        return "Europe"
    if country_code in AMERICAS_CODES:
        return "Americas"
    if country_code in ASIA_CODES:
        return "Asia"
    if country_code in AFRICA_CODES:
        return "Africa"
    if country_code in OCEANIA_CODES:
        return "Oceania"
    return "Other"

def classify_opportunity(text):
    text = str(text).lower()
    for category, words in category_keywords.items():
        pattern = "|".join(re.escape(word) for word in words)
        if re.search(pattern, text):
            return category
    return "Other"



def remove_export_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Remove index columns created by previous CSV exports."""
    return df.loc[:, ~df.columns.str.match(r"^Unnamed:")].copy()


def clean_text(series: pd.Series) -> pd.Series:
    """Trim text and collapse repeated whitespace while retaining nulls."""
    return (
        series.astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .replace("", pd.NA)
    )


def country_name_from_code(value):
    """Translate one or more CORDIS alpha-2 codes into English country names."""
    if pd.isna(value):
        return pd.NA
    codes = [code.strip().upper() for code in re.split(r"[;,]", str(value)) if code.strip()]
    names = []
    for code in codes:
        name = COUNTRY_OVERRIDES.get(code) or ENGLISH.territories.get(code)
        names.append(name or f"Unknown ({code})")
    return "; ".join(names) if names else pd.NA


def split_trial_locations(trials: pd.DataFrame) -> pd.DataFrame:
    """Create one unique trial–country row from CTIS location/status strings."""
    source_column = "Location(s)_and_recruitment_status"
    locations = trials[["Trial_number", source_column]].dropna().copy()
    locations["location_entry"] = locations[source_column].str.split(r"\s*,\s*")
    locations = locations.explode("location_entry", ignore_index=True)

    parts = locations["location_entry"].str.split(":", n=1, expand=True)
    locations["country_name"] = clean_text(parts[0])
    locations["recruitment_status"] = (
        clean_text(parts[1]) if parts.shape[1] > 1 else pd.NA
    )
    locations = locations.dropna(subset=["Trial_number", "country_name"])
    return locations[
        ["Trial_number", "country_name", "recruitment_status"]
    ].drop_duplicates().reset_index(drop=True)


## 3. Clean the CORDIS opportunity table


In [6]:
cordis_raw = remove_export_columns(df_cordis)

print(cordis_raw.columns)

Index(['projectID', 'organisationID', 'name', 'shortName', 'activityType',
       'city', 'country', 'role', 'ecContribution', 'netEcContribution',
       'totalCostOrg', 'endOfParticipation', 'active', 'status', 'title',
       'startDate', 'endDate', 'totalCostProj', 'topics', 'fundingScheme',
       'grantDoi', 'keywords', 'period', 'objective'],
      dtype='str')


In [7]:
text_columns = [
    "projectID", "organisationID","name", "shortName", "activityType", "city", "country",
    "role", "active","status", "title", "topics", "fundingScheme", "keywords", "period", "objective"
]

numeric_columns = [
    "ecContribution", "netEcContribution", "totalCostOrg", "totalCostProj"
]

date_colums =[ "startDate", "endDate", "contentUpdateDate"]

In [8]:
for column in text_columns:
    if column in cordis_raw:
        cordis_raw[column] = clean_text(cordis_raw[column])

for column in numeric_columns:
    if column in cordis_raw:
        cordis_raw[column] = pd.to_numeric(cordis_raw[column], errors="coerce")

for column in date_colums:
    if column in cordis_raw:
        cordis_raw[column] = pd.to_datetime(cordis_raw[column], errors="coerce")

In [9]:
cordis_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 177879 entries, 0 to 177878
Data columns (total 24 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   projectID           177879 non-null  string        
 1   organisationID      177879 non-null  string        
 2   name                177879 non-null  string        
 3   shortName           135365 non-null  string        
 4   activityType        177429 non-null  string        
 5   city                177390 non-null  string        
 6   country             177609 non-null  string        
 7   role                177879 non-null  string        
 8   ecContribution      157588 non-null  float64       
 9   netEcContribution   111061 non-null  float64       
 10  totalCostOrg        177738 non-null  float64       
 11  endOfParticipation  177879 non-null  bool          
 12  active              926 non-null     string        
 13  status              177879 non-null  str

In [10]:
# Adding other features

## Country 
cordis_raw["country_code"] = cordis_raw["country"].str.upper()
cordis_raw["country_name"] = cordis_raw["country_code"].map(country_name_from_code)
cordis_raw["macro_region"] = cordis_raw["country"].map(get_macro_region)

## Datetime
cordis_raw["project_start_year"] = cordis_raw["startDate"].dt.year.astype("Int64")
cordis_raw["project_end_year"] = cordis_raw["endDate"].dt.year.astype("Int64")

## Classification
cordis_raw["category"] = cordis_raw["objective"].apply(classify_opportunity)

In [12]:
cordis_raw["organisation_project_key"] = (
    cordis_raw["organisationID"].astype("string") + "|" + cordis_raw["projectID"].astype("string")
)

before = len(cordis_raw)
cordis_raw = cordis_raw.drop_duplicates().reset_index(drop=True)
print(f"CORDIS rows: {before:,} -> {len(cordis_raw):,} ({before - len(cordis_raw):,} exact duplicates removed)")
print(f"Projects: {cordis_raw['projectID'].nunique():,}")
print(f"Countries: {cordis_raw['country_name'].nunique():,}")
cordis_raw[["country_code", "country_name"]].drop_duplicates().sort_values("country_name").head(15)

CORDIS rows: 177,879 -> 177,866 (13 exact duplicates removed)
Projects: 32,779
Countries: 190


,country_code,country_name
49409,AF,Afghanistan
11063,AL,Albania
5725,DZ,Algeria
143231,AD,Andorra
30223,AO,Angola
23250,AI,Anguilla
2471,AR,Argentina
4703,AM,Armenia
126093,AW,Aruba
163,AU,Australia


## 4. Clean and normalize the CTIS trial table

In [14]:
df_trials.head(2)

,Unnamed: 0,Trial_number,Protocol_code,Title_of_the_trial,Overall_trial_status,Location(s)_and_recruitment_status,Age_group,Age_range_secondary_identifier,Gender,Number_of_participants_enrolled,...,Primary_endpoint,Secondary_endpoints,Decision_date,Start_date,End_date,Global_end_of_the_trial,Trial_results,Sponsor/Co-Sponsors,Sponsor_type,Last_updated
0,0,2023-509723-41-00,REALL_ CART,A Phase I Clinical Trial of CART cell therapy ...,Not authorised,Spain:Not authorised,"18-64 years, 0-17 years",NaN,"Female, Male",10,...,"In this umbrella trial, we will determine in p...",Expression of CD19/CD22 and NKG2DL on primary ...,2024-07-05,NaN,NaN,NaN,No,Fundacion Para La Investigacion Biomedica Del ...,Patient organisation/association,2024-07-05
1,2,2022-501417-31-00,MK-7684A-010,"A Phase 3, Randomized, Double-blind, Active-Co...",Not authorised,"Poland:Not authorised, Italy:Not authorised, G...","18-64 years, 65+ years, 0-17 years","12-17 years, 65-84 years, 85+ years, N/A","Female, Male",510,...,Recurrence-Free Survival (RFS),"Distant Metastasis-Free Survival (DMFS), Overa...",2023-02-03,NaN,NaN,NaN,No,Merck Sharp & Dohme LLC,Pharmaceutical company,2023-06-03


In [15]:
df_trials.columns

Index(['Unnamed: 0', 'Trial_number', 'Protocol_code', 'Title_of_the_trial',
       'Overall_trial_status', 'Location(s)_and_recruitment_status',
       'Age_group', 'Age_range_secondary_identifier', 'Gender',
       'Number_of_participants_enrolled', 'Trial_region', 'Medical_conditions',
       'Therapeutic_area', 'Trial_phase', 'Product', 'Primary_endpoint',
       'Secondary_endpoints', 'Decision_date', 'Start_date', 'End_date',
       'Global_end_of_the_trial', 'Trial_results', 'Sponsor/Co-Sponsors',
       'Sponsor_type', 'Last_updated'],
      dtype='str')

In [ ]:
trials = remove_export_columns(df_trials)

for column in trials.select_dtypes(include=["object"]).columns:
    trials[column] = clean_text(trials[column])

trial_date_columns = [
    "Decision_date", "Start_date", "End_date", "Global_end_of_the_trial", "Last_updated",
]
for column in trial_date_columns:
    if column in trials:
        trials[column] = pd.to_datetime(trials[column], errors="coerce")

trials["Number_of_participants_enrolled"] = pd.to_numeric(
    trials["Number_of_participants_enrolled"], errors="coerce"
).astype("Int64")

before = len(trials)
trials = trials.drop_duplicates(subset=["Trial_number"], keep="first").reset_index(drop=True)
trial_countries = split_trial_locations(trials)

country_rollup = (
    trial_countries.groupby("Trial_number")["country_name"]
    .agg(lambda values: "; ".join(sorted(set(values))))
    .rename("country_names")
)
country_count = trial_countries.groupby("Trial_number")["country_name"].nunique().rename("country_count")
trials = trials.merge(country_rollup, on="Trial_number", how="left")
trials = trials.merge(country_count, on="Trial_number", how="left")
trials["country_count"] = trials["country_count"].fillna(0).astype("Int64")
trials["decision_year"] = trials["Decision_date"].dt.year.astype("Int64")

print(f"Trial rows: {before:,} -> {len(trials):,} ({before - len(trials):,} duplicate trial IDs removed)")
print(f"Trial–country relationships: {len(trial_countries):,}")
print(f"Trial countries: {trial_countries['country_name'].nunique():,}")
trial_countries.head(10)

## 5. Data-quality checks

In [ ]:
assert trials["Trial_number"].is_unique, "Trial_number must be unique after cleaning."
assert not trial_countries.duplicated(["Trial_number", "country_name"]).any(), "Duplicate trial-country pair found."
assert consur["projectID"].notna().all(), "CORDIS contains missing project IDs."

quality_report = pd.DataFrame([
    {
        "dataset": "CORDIS",
        "rows": len(consur),
        "unique_entities": consur["projectID"].nunique(),
        "missing_country": consur["country_name"].isna().sum(),
    },
    {
        "dataset": "CTIS trials",
        "rows": len(trials),
        "unique_entities": trials["Trial_number"].nunique(),
        "missing_country": trials["country_names"].isna().sum(),
    },
])
quality_report

## 6. Export dashboard-ready tables

In [ ]:
consur.to_csv("../../_local_data_backup/ConsurDatabase_clean.csv")
trials.to_csv("../Scripts/data/data_final/TrialsDatabase_clean.csv")
trial_countries.to_csv("../Scripts/data/data_final/TrialCountry_clean.csv")